# 01 - What LLiMa Is And Where It Sits

**Why this notebook first.** Before you pull a model or write a line of `pyneat.genai` code, you
need a correct mental model of two things that are easy to conflate: the **LLiMa CLI** (which
*prepares* generative models) and the **`pyneat.genai` API** (which *runs* them from your app). Get
this split right and everything else in the folder follows.

This notebook is concept-only. It has no heavy cells - the code cells here just print facts and do
small string checks, so they are safe to run anywhere. The model-running notebooks are in
`../02-run-llm-vlm/`.

## What LLiMa is

**LLiMa** is SiMa's toolchain for **generative** models - LLMs (text), VLMs (image + text), and ASR
(speech-to-text) - on Modalix. It owns three jobs:

1. **Model preparation / download** - fetch a model that has already been quantized and compiled for
   Modalix from a remote catalog, and lay it down as a *model directory* on the board.
2. **Command-line testing** - a quick interactive `run` to sanity-check a model without writing code.
3. **Benchmarking** - `benchmark-server` starts an HTTP server used to measure a model's throughput.

On the DevKit the CLI is a single binary at **`/usr/bin/llima`**. It is a Python entry point
(`sima_lmm.devkit.devkit_demo`); it is **not** in the SDK container, only on the board.

The key output of LLiMa is a **model directory** under `/media/nvme/llima/models/<model-id>`. That
directory is the hand-off point: LLiMa produces it, and the `pyneat.genai` API consumes it.

> Source of truth: `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`
> ("LLiMa owns GenAI model preparation, command-line testing, and benchmarking. Neat owns the
> application-facing API and runtime integration once the model is used inside your app.")

## Where LLiMa sits in the NEAT stack

NEAT has two model paths that look similar but do not overlap:

| Path | For | Load with | Run with |
| --- | --- | --- | --- |
| **Classic `Model`** | Fixed-shape *discriminative* models: classification, detection, segmentation, embeddings (e.g. YOLO). | `pyneat.Model("model.tar.gz")` | `Graph` / `Run` (see `../../tutorial`) |
| **GenAI / LLiMa** | *Generative* models: LLM, VLM, ASR. | `pyneat.genai.GenAIModel("<model dir>")` | `model.run(request)` / `model.stream(request)` |

Rule of thumb from the docs: use the classic `Model` API for fixed-shape discriminative models; use
the GenAI APIs when your application asks a model to **generate** text, answer questions about
images, call tools, or transcribe audio.

The high-level GenAI shape (identical in C++ and Python):

1. Prepare or download a model artifact for Modalix (**LLiMa**).
2. Put it on the target under `/media/nvme/llima/models/`.
3. Load it from your app (**`pyneat.genai`**).
4. Send a `GenerationRequest`.
5. Read a `GenerationResult`, or consume a `GenerationStream`.

## Supported model families

`llima search` returns a catalog of models already prepared for Modalix. They fall into three
families. The code cell below embeds that catalog as text - safe to run, it only prints and counts
strings.

In [ ]:
# Safe cell: the `llima search` catalog embedded as text (no network, no model run).
CATALOG = """
llava-1.5-7b-hf-a16w4
Llama-3.1-8B-Instruct-a16w4
Llama-3.2-3B-Instruct-a16w4
Phi-3.5-mini-instruct-a16w4
paligemma-3b-pt-224-a16w8
gemma-3-4b-it-a16w4
Llama-2-7b-chat-hf-a16w4
whisper-small-a16w8
gemma-3-1b-it-a16w4
Mistral-7B-Instruct-v0.3-a16w4
gemma3-siglip448-a16w4
Qwen2.5-0.5B-Instruct-GPTQ-a16w4
Qwen2.5-7B-Instruct-GPTQ-a16w4
Qwen3-0.6B-GPTQ-a16w4
Qwen3-1.7B-GPTQ-a16w4
Qwen3-4B-Instruct-2507-GPTQ-a16w4
Qwen3-8B-GPTQ-a16w4
LFM2-VL-450M-a16w4
LFM2-VL-1.6B-a16w4
LFM2-VL-3B-a16w4
Qwen2.5-VL-3B-Instruct-GPTQ-a16w4
Qwen3-VL-4B-Instruct-GPTQ-a16w4
Qwen3-VL-8B-Instruct-GPTQ-a16w4
LFM2-350M-a16w4
LFM2-1.2B-a16w4
LFM2-2.6B-a16w4
Qwen2.5-VL-7B-Instruct-GPTQ-a16w4
Qwen3-VL-2B-Instruct-GPTQ-a16w4
gemma-4-E2B-it-GPTQ-a16w4
gemma-4-E4B-it-GPTQ-a16w4
LFM2.5-230M-a16w4
LFM2.5-350M-a16w4
LFM2.5-1.2B-Instruct-a16w4
LFM2.5-1.2B-Thinking-a16w4
LFM2.5-VL-450M-a16w4
LFM2.5-VL-1.6B-a16w4
""".split()

vlm = [m for m in CATALOG if "-VL-" in m or m.startswith(("llava", "paligemma", "gemma3-siglip"))]
asr = [m for m in CATALOG if m.startswith("whisper")]
llm = [m for m in CATALOG if m not in vlm and m not in asr]

print(f"catalog size: {len(CATALOG)} models")
print(f"  LLM (text):        {len(llm)}  e.g. {llm[:3]}")
print(f"  VLM (image+text):  {len(vlm)}  e.g. {vlm[:3]}")
print(f"  ASR (speech->text):{len(asr):>3}  e.g. {asr}")

**Interpretation.** The catalog is dominated by decoder LLMs (Qwen3, Llama, Mistral, Phi,
Gemma, LFM2), a strong set of VLMs (the `*-VL-*` families plus llava / paligemma / siglip), and a
single ASR model (`whisper-small-a16w8`). Family cheat-sheet:

- **LLM** - text in, text out. Chat, summarization, tool-calling.
- **VLM** - image(s) + text in, text out. Captioning, visual Q&A, document reading.
- **ASR** - audio in, text out. Transcription.

Every model ID ends in a **quantization suffix** - the next cell decodes it.

## Quantization suffixes (a correctness trap)

The suffix after the model name is the LLiMa quantization scheme. It is not decoration - it tells you
the weight precision, which drives the on-disk size and the memory footprint.

- **`a16w4`** = **16-bit activations / 4-bit weights** (the common case; smallest weights).
- **`a16w8`** = **16-bit activations / 8-bit weights** (e.g. `whisper-small-a16w8`,
  `paligemma-3b-pt-224-a16w8`).

`a16w8` weights are roughly twice the size of `a16w4` weights for the same parameter count, all else
equal.

In [ ]:
def decode_suffix(model_id: str) -> str:
    if model_id.endswith("a16w4"):
        return "16-bit activations / 4-bit weights"
    if model_id.endswith("a16w8"):
        return "16-bit activations / 8-bit weights"
    return "unknown quantization"

for m in ["Qwen3-4B-Instruct-2507-GPTQ-a16w4", "whisper-small-a16w8"]:
    print(f"{m:45s} -> {decode_suffix(m)}")

**Interpretation.** `Qwen3-4B-...-a16w4` keeps its 4B weights at 4 bits each; `whisper-small`
keeps its weights at 8 bits. When you compare two models, read the suffix before you read the
parameter count - an `a16w8` model of half the parameters can still be as heavy on disk as an `a16w4`
model with more parameters.

## Memory and size constraints (the real limiter on a DevKit)

Two independent budgets constrain GenAI on the DevKit, and both are tight:

**1. Disk (before you can even load a model).** The DevKit root filesystem is small, and a 4B `a16w4`
LLM plus a 4B `a16w4` VLM already crowd it. So:

- **Always run `df -h /` before `llima pull`** to check free space first. A failed half-download
  wastes both time and disk.
- Prefer the smallest viable variant when pulling fresh: `LFM2.5-230M-a16w4` /
  `Qwen3-0.6B-GPTQ-a16w4` (LLM), `LFM2-VL-450M-a16w4` (VLM).
- The happy-path models used in this folder are already deployed, so no pull is needed for them.

**2. Runtime memory (while a model is loaded).** Each loaded model holds its weights plus a KV cache
that grows with context length. Loading an LLM and a VLM at the same time means both weight sets are
resident at once. Practical guidance:

- Keep only the message history your app needs - long histories grow the KV cache and the
  time-to-first-token.
- One process serving multiple models (via `GenAIServer`) still shares the single MLA hardware
  gatekeeper; running two server processes does **not** multiply throughput, it just doubles the
  memory pressure.

In [ ]:
# Safe cell: the three happy-path models this folder uses (as reported by `llima list`).
ON_BOARD = {
    "LLM": "Qwen3-4B-Instruct-2507-GPTQ-a16w4",
    "VLM": "Qwen3-VL-4B-Instruct-GPTQ-a16w4",
    "ASR": "whisper-small-a16w8",
}
MODELS_ROOT = "/media/nvme/llima/models"
for role, mid in ON_BOARD.items():
    print(f"{role}: {MODELS_ROOT}/{mid}")

SMALLEST_FRESH = ["LFM2.5-230M-a16w4", "Qwen3-0.6B-GPTQ-a16w4", "LFM2-VL-450M-a16w4"]
print("\nsmallest viable if pulling fresh:", SMALLEST_FRESH)

## Expected output

Running the safe cells above (they touch no hardware) prints:

```text
catalog size: 36 models
  LLM (text):        27  e.g. ['Llama-3.1-8B-Instruct-a16w4', 'Llama-3.2-3B-Instruct-a16w4', 'Phi-3.5-mini-instruct-a16w4']
  VLM (image+text):  8  e.g. ['llava-1.5-7b-hf-a16w4', 'paligemma-3b-pt-224-a16w8', 'gemma3-siglip448-a16w4']
  ASR (speech->text):  1  e.g. ['whisper-small-a16w8']
```

(Exact per-family counts depend only on the classification heuristic in the cell; the catalog total
is 36.)

```text
Qwen3-4B-Instruct-2507-GPTQ-a16w4             -> 16-bit activations / 4-bit weights
whisper-small-a16w8                           -> 16-bit activations / 8-bit weights
```

```text
LLM: /media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4
VLM: /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4
ASR: /media/nvme/llima/models/whisper-small-a16w8

smallest viable if pulling fresh: ['LFM2.5-230M-a16w4', 'Qwen3-0.6B-GPTQ-a16w4', 'LFM2-VL-450M-a16w4']
```

## Recap and next

- **LLiMa (CLI)** prepares a *model directory*; **`pyneat.genai`** runs it. Do not confuse the two.
- Three families: **LLM / VLM / ASR**. One ASR model in the catalog (`whisper-small-a16w8`).
- Read the **quantization suffix** (`a16w4` vs `a16w8`) before the parameter count.
- The DevKit is **disk- and memory-constrained**: `df -h /` before every `llima pull`.

Next: **`02_llima_cli.ipynb`** walks each `llima` subcommand with its real captured output and what
it does on disk. After that, `../02-run-llm-vlm/` runs models from Python.

## References

- `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`
- `/workspace/core/tutorials/019_run_an_llm/README.md`
- Official docs: developer.sima.ai/software/getting-started/ and .../genai-llima/runtime